# Datadelen i Kunstpixel: kjør og sjekk i Colab

Denne notatboken lar deg (eller en språkmodell) se på og sjekke dataene uten VS Code eller andre verktøy. Den virker i Google Colab og lokalt i Jupyter.

**Hva den gjør:** finner repoet, installerer det som trengs, viser tabellene (kun lesing), gjør et prøvebygg og kjører statusporten.

**Hva den aldri gjør:** den bygger ikke `05_data/museum.db` på Drive. Den filen bygges bare av Lars. Les `docs/DATAPIPELINE.md` først.

Kjør cellene i rekkefølge.

## 1. Finn repoet

I Colab monteres Google Drive. Mappen «Min disk» heter `/content/drive/MyDrive`. Endre stien hvis repoet ligger et annet sted.

In [ ]:
from pathlib import Path
import os, sys

try:
    from google.colab import drive  # finnes bare i Colab
    drive.mount('/content/drive')
    PROSJEKT = Path('/content/drive/MyDrive/Kunstpixel')  # <- endre om repoet ligger et annet sted på Drive
except ImportError:
    PROSJEKT = Path.cwd()  # lokalt: start fra rotmappen i repoet
    if PROSJEKT.name == 'notebooks':
        PROSJEKT = PROSJEKT.parent

assert (PROSJEKT / '05_data' / 'museum.db').is_file(), f'Fant ikke repoet i {PROSJEKT}. Sett PROSJEKT riktig.'
os.chdir(PROSJEKT)
sys.path.insert(0, str(PROSJEKT))
print('Prosjekt:', PROSJEKT)

## 2. Avhengigheter

Må være `mcp==1.29.1`. Versjon 2.x virker ikke (FastMCP er omdøpt).

In [ ]:
import importlib.metadata as md_, importlib.util, subprocess

KRAV = [('mcp', '1.29.1'), ('pydantic', None), ('PyYAML', None), ('pytest', None)]
def mangler(pakke, versjon):
    try:
        return versjon is not None and md_.version(pakke) != versjon
    except md_.PackageNotFoundError:
        return True

installer = [f'{p}=={v}' if v else p for p, v in KRAV if mangler(p, v)]
if installer:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *installer], check=True)
print('Installerte:', installer if installer else 'ingenting nytt, alt var på plass')

## 3. Se på dataene (kun lesing)

In [ ]:
import sqlite3

con = sqlite3.connect((PROSJEKT / '05_data' / 'museum.db').resolve().as_uri() + '?mode=ro', uri=True)
tabeller = [r[0] for r in con.execute("select name from sqlite_master where type='table' and name not like 'sqlite_%' order by name")]
for t in tabeller:
    print(f'{t:34}{con.execute(f"select count(*) from {t}").fetchone()[0]:>9}')

In [ ]:
import pandas as pd

# Reglene som velger vurdering og forslag i rapportene. Redigeres i 05_data/rapport_regler.csv, ikke her.
pd.read_sql('select regel_id, sporsmal_id, felt, operator, terskel, vurdering from rapport_regler order by sporsmal_id, regel_id', con)

## 4. Prøvebygg: stemmer museum.db med datafilene?

Bygger til en midlertidig fil og sammenligner hashen. `museum.db` røres ikke. «ULIK» betyr at datafilene er endret siden sist Lars bygget.

In [ ]:
import tempfile
from src.kunstpixel.data import build_museum_db as bygger

with tempfile.TemporaryDirectory() as tmp:
    ut = Path(tmp) / 'proebygg.db'
    info = bygger.bygg(ut)
    ny = bygger.sha256(ut)
gjeldende = (PROSJEKT / '05_data' / 'museum.db.sha256').read_text().split()[0]
print(info)
print('Prøvebygg:', ny[:12], '| museum.db:', gjeldende[:12], '->', 'LIK' if ny == gjeldende else 'ULIK: museum.db må bygges på nytt av Lars')

## 5. Datafilene er hele

Fanger komma uten anførselstegn og feil antall kolonner i CSV-filene.

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_data_csv.py', '-q', '-p', 'no:cacheprovider'], capture_output=True, text=True)
print(r.stdout[-1500:])
print('exit:', r.returncode, '(0 = alt bestått)')

## 6. Statusporten

Hele sjekken: to bygg gir lik hash, integritet, syntetisk merking, tracer-sjekker mot verktøyene og hullene fra briefen. Sett `MED_TESTER = True` for å kjøre alle testene også (tar flere minutter).

In [ ]:
MED_TESTER = False

kommando = [sys.executable, 'scripts/db_status_gate.py'] + ([] if MED_TESTER else ['--uten-pytest'])
r = subprocess.run(kommando, capture_output=True, text=True)
print(r.stdout[-3500:])
print('exit:', r.returncode, '(0 = KLAR for sammenslåing)')

## Slik endrer du data

1. Rediger CSV-en i `05_data/` (eller generatoren i `src/kunstpixel/data/archive_generator.py`). Aldri `museum.db` direkte.
2. Kjør celle 5 og 4: CSV-en skal være hel, og prøvebygget viser at noe er endret.
3. Kjør celle 6. Skal gi KLAR.
4. Be Lars bygge `museum.db` (`python src/kunstpixel/data/build_museum_db.py`), og lag så demokopien (`build_demo_db.py`).

Hele oppskriften og reglene står i `docs/DATAPIPELINE.md`.